# Construcción de rezagos y medias móviles ambientales 2024

1. **Objetivo del notebook**

- Crear rezagos de 1, 2 y 4 semanas y medias móviles de 4 semanas para las variables ambientales dinámicas del panel integrado.

2. **Decisiones metodológicas**

- Calcular las variables por `grid_id`, ordenadas por `year` y `epiweek`.
- Usar la semana actual y las tres semanas anteriores para `rolling_mean_4`.
- Mantener como nulos los rezagos que no tienen suficientes semanas previas y las medias móviles incompletas.
- No calcular variables derivadas para altitud, población, clasificación de Köppen ni componentes estacionales.

3. **Fuente**

- `data/processed/integrated/analytical_grid_week_2024.parquet`.

4. **Producto**

- `data/processed/integrated/analytical_grid_week_2024_lags_n_rolls.parquet`.


In [1]:
from pathlib import Path
import pandas as pd

for candidate in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (candidate / 'src' / 'paths.py').is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('No se encontró src/paths.py.')

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.plot_palette import COLORS, MAP_CMAP, apply_plot_palette
apply_plot_palette()

INPUT_PATH = PROJECT_ROOT / 'data/processed/integrated/analytical_grid_week_2024.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data/processed/integrated/analytical_grid_week_2024_lags_n_rolls.parquet'
DYNAMIC_ENVIRONMENTAL = [
    'tmean_c', 'tmin_c', 'tmax_c',
    'rh_mean', 'rh_min', 'rh_max', 'tp_sum',
    'ndvi_mean_biweekly', 'ndvi_max_biweekly',
    'ndvi_min_biweekly', 'ndvi_std_biweekly',
]

df = pd.read_parquet(INPUT_PATH).sort_values(['grid_id', 'year', 'epiweek']).copy()
missing = [column for column in DYNAMIC_ENVIRONMENTAL if column not in df.columns]
if missing:
    raise KeyError(f'Faltan variables ambientales: {missing}')

grouped = df.groupby('grid_id', sort=False)
for column in DYNAMIC_ENVIRONMENTAL:
    for lag in [1, 2, 4]:
        df[f'{column}_lag_{lag}'] = grouped[column].shift(lag)
    df[f'{column}_rolling_mean_4'] = grouped[column].transform(
        lambda values: values.rolling(window=4, min_periods=4).mean()
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)
print(f'Parquet guardado: {OUTPUT_PATH} ({len(df):,} filas, {len(df.columns):,} columnas)')


Parquet guardado: C:\tesis_aedes_grilla_2024\data\processed\integrated\analytical_grid_week_2024_lags_n_rolls.parquet (177,268 filas, 73 columnas)


## Vista del parquet generado

Primeras 10 filas del producto con rezagos y medias móviles.

In [2]:
display(pd.read_parquet(OUTPUT_PATH).head(10))


,grid_id,year,epiweek,tmean_c,tmin_c,tmax_c,elevation_min,elevation_mean,elevation_max,rh_mean,...,ndvi_max_biweekly_lag_4,ndvi_max_biweekly_rolling_mean_4,ndvi_min_biweekly_lag_1,ndvi_min_biweekly_lag_2,ndvi_min_biweekly_lag_4,ndvi_min_biweekly_rolling_mean_4,ndvi_std_biweekly_lag_1,ndvi_std_biweekly_lag_2,ndvi_std_biweekly_lag_4,ndvi_std_biweekly_rolling_mean_4
0,0,2024,1,21.209301,19.992899,22.276896,25.0,33.596774,43.0,74.356583,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,2024,2,22.054096,21.573660,22.765982,25.0,33.596774,43.0,73.982315,...,NaN,NaN,-0.021624,NaN,NaN,NaN,0.004055,NaN,NaN,NaN
2,0,2024,3,22.568083,22.038534,23.052988,25.0,33.596774,43.0,75.665398,...,NaN,NaN,-0.004966,-0.021624,NaN,NaN,0.003381,0.004055,NaN,NaN
3,0,2024,4,23.774378,23.398520,24.111084,25.0,33.596774,43.0,72.535446,...,NaN,0.078348,0.011691,-0.004966,NaN,-0.002519,0.002707,0.003381,NaN,0.002874
4,0,2024,5,22.861515,22.499634,23.158884,25.0,33.596774,43.0,73.460915,...,0.081831,0.073279,0.004822,0.011691,-0.021624,0.002375,0.001353,0.002707,0.004055,0.001860
5,0,2024,6,25.018829,24.110687,25.648270,25.0,33.596774,43.0,71.713272,...,0.080882,0.113707,-0.002047,0.004822,-0.004966,-0.036961,0.000000,0.001353,0.003381,0.001184
6,0,2024,7,23.730053,23.066925,24.236368,25.0,33.596774,43.0,75.282051,...,0.079934,0.172932,-0.162309,-0.002047,0.011691,-0.074957,0.000676,0.000000,0.002707,0.000507
7,0,2024,8,24.646017,23.532104,25.408484,25.0,33.596774,43.0,76.529297,...,0.070744,0.236070,-0.140295,-0.162309,0.004822,-0.113702,0.000000,0.000676,0.001353,0.000169
8,0,2024,9,23.541357,22.868347,24.530279,25.0,33.596774,43.0,75.064690,...,0.061555,0.236288,-0.150157,-0.140295,-0.002047,-0.118250,0.000000,0.000000,0.000000,0.000169
9,0,2024,10,23.356249,22.439676,24.117676,25.0,33.596774,43.0,74.060120,...,0.242594,0.192981,-0.020239,-0.150157,-0.162309,-0.082152,0.000000,0.000000,0.000676,0.000916
